# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Make a copy to avoid modifying the original DataFrame directly
df = spaceship.copy()

# Separate target variable
X = df.drop('Transported', axis=1)
y = df['Transported']

In [5]:
# Feature Engineering for 'Cabin' column
# Split 'Cabin' into 'Cabin_deck', 'Cabin_num', 'Cabin_side'
X[['Cabin_deck', 'Cabin_num', 'Cabin_side']] = X['Cabin'].str.split('/', expand=True)
X = X.drop('Cabin', axis=1)

# Convert 'Cabin_num' to numeric, coerce errors to NaN
X['Cabin_num'] = pd.to_numeric(X['Cabin_num'], errors='coerce')

In [6]:
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Remove 'PassengerId' and 'Name' from features if they are present
if 'PassengerId' in numerical_features:
    numerical_features.remove('PassengerId')
if 'PassengerId' in categorical_features:
    categorical_features.remove('PassengerId')
if 'Name' in categorical_features:
    categorical_features.remove('Name')

# Define preprocessing steps
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep other columns (like PassengerId for submission if needed)
)

# Apply preprocessing
X_preprocessed = preprocessor.fit_transform(X)

# Get feature names after one-hot encoding for categorical features and including passthrough columns
all_feature_names = preprocessor.get_feature_names_out()

# Convert the preprocessed data back to a DataFrame
X_processed_df = pd.DataFrame(X_preprocessed, columns=all_feature_names)

print("Shape of preprocessed data:", X_processed_df.shape)
display(X_processed_df.head())

Shape of preprocessed data: (8693, 29)


,num__Age,num__RoomService,num__FoodCourt,num__ShoppingMall,num__Spa,num__VRDeck,num__Cabin_num,cat__HomePlanet_Earth,cat__HomePlanet_Europa,cat__HomePlanet_Mars,...,cat__Cabin_deck_C,cat__Cabin_deck_D,cat__Cabin_deck_E,cat__Cabin_deck_F,cat__Cabin_deck_G,cat__Cabin_deck_T,cat__Cabin_side_P,cat__Cabin_side_S,remainder__PassengerId,remainder__Name
0,0.709437,-0.34059,-0.287314,-0.290817,-0.276663,-0.269023,-1.186627,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0001_01,Maham Ofracculy
1,-0.336717,-0.175364,-0.281669,-0.248968,0.211505,-0.230194,-1.186627,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0002_01,Juanna Vines
2,2.034566,-0.275409,1.955616,-0.290817,5.694289,-0.225782,-1.186627,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0003_01,Altark Susent
3,0.290975,-0.34059,0.517406,0.330225,2.683471,-0.098708,-1.186627,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0003_02,Solam Susent
4,-0.894666,0.118709,-0.243409,-0.038048,0.225732,-0.267258,-1.184651,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0004_01,Willy Santantines


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_processed_df, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (6954, 29)
X_test shape: (1739, 29)
y_train shape: (6954,)
y_test shape: (1739,)


In [12]:
# Drop non-numerical 'remainder' columns from X_train before fitting
X_train_cleaned = X_train.drop(columns=['remainder__PassengerId', 'remainder__Name'], errors='ignore')

# Drop non-numerical 'remainder' columns from X_test before making predictions
X_test_cleaned = X_test.drop(columns=['remainder__PassengerId', 'remainder__Name'], errors='ignore')

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
#your code here

In [13]:
print('\n--- Gradient Boosting Regressor ---')

# Initialize and train the Gradient Boosting Regressor
gradient_boosting_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gradient_boosting_reg.fit(X_train_cleaned, y_train)

# Make predictions on the cleaned test set
y_pred_gb = gradient_boosting_reg.predict(X_test_cleaned)

# Evaluate the Gradient Boosting model
r2_gb = r2_score(y_test, y_pred_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
mse_gb = mean_squared_error(y_test, y_pred_gb)
rmse_gb = root_mean_squared_error(y_test, y_pred_gb)

print(f"R2 Score (Gradient Boosting): {r2_gb:.4f}")
print(f"Mean Absolute Error (MAE) (Gradient Boosting): {mae_gb:.4f}")
print(f"Mean Squared Error (MSE) (Gradient Boosting): {mse_gb:.4f}")
print(f"Root Mean Squared Error (RMSE) (Gradient Boosting): {rmse_gb:.4f}")


--- Gradient Boosting Regressor ---
R2 Score (Gradient Boosting): 0.4682
Mean Absolute Error (MAE) (Gradient Boosting): 0.2812
Mean Squared Error (MSE) (Gradient Boosting): 0.1329
Root Mean Squared Error (RMSE) (Gradient Boosting): 0.3646


- Evaluate your model

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

- Run Grid Search

- Evaluate your model

In [15]:
from sklearn.model_selection import GridSearchCV

print('\n--- Ajusting Hyperparameters for Gradient Boosting ---')

# Defining the parameters to look for
param_grid = {
    'n_estimators': [50, 100, 200], # Número de árboles
    'learning_rate': [0.01, 0.1, 0.2], # Tasa de aprendizaje
    'max_depth': [3, 4, 5] # Profundidad máxima de cada árbol
}

# Initializing Model Gradient Boosting
gradient_boosting_reg = GradientBoostingRegressor(random_state=42)

# Configuring GridSearchCV
# cv=5 indicates 5-fold cross-validation
# scoring='r2' to optimize the meter for R2
grid_search = GridSearchCV(estimator=gradient_boosting_reg, param_grid=param_grid,
                           cv=5, scoring='r2', n_jobs=-1, verbose=1)

# Ejecuting the search of the fit in the clean data train
grid_search.fit(X_train_cleaned, y_train)

print(f"Mejores hiperparámetros encontrados: {grid_search.best_params_}")
print(f"Mejor puntuación R2 (en validación cruzada): {grid_search.best_score_:.4f}")

# Getting the best model
best_gb_model = grid_search.best_estimator_

# Predctions of the best model in test data
y_pred_gb_tuned = best_gb_model.predict(X_test_cleaned)

# Evaluating the best model
r2_gb_tuned = r2_score(y_test, y_pred_gb_tuned)
mae_gb_tuned = mean_absolute_error(y_test, y_pred_gb_tuned)
mse_gb_tuned = mean_squared_error(y_test, y_pred_gb_tuned)
rmse_gb_tuned = root_mean_squared_error(y_test, y_pred_gb_tuned)

print(f"\n--- Evaluación del Mejor Modelo Gradient Boosting ---")
print(f"R2 Score (Gradient Boosting Ajustado): {r2_gb_tuned:.4f}")
print(f"Mean Absolute Error (MAE) (Gradient Boosting Ajustado): {mae_gb_tuned:.4f}")
print(f"Mean Squared Error (MSE) (Gradient Boosting Ajustado): {mse_gb_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE) (Gradient Boosting Ajustado): {rmse_gb_tuned:.4f}")


--- Ajusting Hyperparameters for Gradient Boosting ---
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Mejores hiperparámetros encontrados: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}
Mejor puntuación R2 (en validación cruzada): 0.4814

--- Evaluación del Mejor Modelo Gradient Boosting ---
R2 Score (Gradient Boosting Ajustado): 0.4803
Mean Absolute Error (MAE) (Gradient Boosting Ajustado): 0.2712
Mean Squared Error (MSE) (Gradient Boosting Ajustado): 0.1299
Root Mean Squared Error (RMSE) (Gradient Boosting Ajustado): 0.3604
